# 第1回 定常性と自己相関

このノートブックは、スライドと同じ順に並んでいます。
コードはすべて完成しているので、上から順に実行して、結果を読んでください。

- 0節：セットアップ（授業の最初に実行する）
- A節：理論をシミュレーションで確かめる
- B節：実データを解析する
- C節：設定を変えて回し直す

今日の問い：時間の順に並んだデータに、いつもの統計（回帰・標本平均）をそのまま使うと、何が起きるか。

## 0. セットアップ

授業が始まったら、まずこのセルを実行してください。
パッケージのインストールに少し時間がかかります。終わるまで、次のセルは動きません。

In [ ]:
# OSのコードネームを確認（noble = Ubuntu 24.04 など）
system("lsb_release -cs", intern = TRUE)

options(
  repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/noble/2026-09-01"),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
    paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"]))
)
install.packages(c("tsibble", "tsibbledata", "feasts"))
sessionInfo()

## A. 理論をシミュレーションで確かめる

### A-1. 見せかけの回帰

互いに無関係なランダムウォーク（random walk）を2本作り、片方をもう片方に回帰します。
これを1000回くり返し、傾きのp値が0.05を下回った割合を出します。

実行する前に、割合を予想してください。2本は無関係なので、ふつうの検定なら5%になるはずです。

ふつうの検定は、観測が互いに独立で、同じ分布に従うこと（独立同分布、independent and identically distributed, iid）を前提にしています。

In [ ]:
set.seed(1)
p <- replicate(1000, {
  x <- cumsum(rnorm(200)); y <- cumsum(rnorm(200))
  summary(lm(y ~ x))$coefficients[2, 4]
})
mean(p < 0.05)

出てきた数が、無関係な2本の系列で「有意」になった割合です。

比べるために、足し合わせる前の乱数で同じことをします。こちらはiidです。
違いは`cumsum()`を外したことだけです。

In [ ]:
set.seed(1)
p_iid <- replicate(1000, {
  x <- rnorm(200); y <- rnorm(200)
  summary(lm(y ~ x))$coefficients[2, 4]
})
mean(p_iid < 0.05)

こちらは、ほぼ5%になります。検定が壊れたのではなく、iidでないデータに、iid前提の検定を使ったことが原因です。

データを増やせば直るかを確かめます。系列の長さを50、200、800と変えます（くり返しは各500回）。

In [ ]:
set.seed(1)
rate <- sapply(c(50, 200, 800), function(n) {
  p <- replicate(500, {
    x <- cumsum(rnorm(n)); y <- cumsum(rnorm(n))
    summary(lm(y ~ x))$coefficients[2, 4]
  })
  mean(p < 0.05)
})
data.frame(n = c(50, 200, 800), rate = rate)

長くするほど、割合は上がります。データを増やしても直りません。

### A-2. 3つの系列と自己相関

ここからパッケージを使います。0節のインストールが終わっていることを確かめてください。

In [ ]:
library(tsibble)
library(tsibbledata)
library(feasts)
library(ggtime)   # ACFの図（autoplot）
library(dplyr)
library(ggplot2)
options(repr.plot.width = 8, repr.plot.height = 4.5)

ホワイトノイズ（white noise）、AR(1)過程、ランダムウォークを、同じ乱数から作ります。

- ホワイトノイズ：$x_t = w_t$
- AR(1)：$x_t = 0.7\,x_{t-1} + w_t$
- ランダムウォーク：$x_t = x_{t-1} + w_t$

In [ ]:
set.seed(1)
n <- 500
w <- rnorm(n)
sim <- tsibble(
  t = rep(1:n, 3),
  series = rep(c("1 white noise", "2 AR(1)", "3 random walk"), each = n),
  x = c(w, as.numeric(stats::filter(w, 0.7, method = "recursive")), cumsum(w)),
  key = series, index = t
)
ggplot(sim, aes(t, x)) +
  geom_line() +
  facet_wrap(~series, ncol = 1, scales = "free_y") +
  labs(x = "Time t (step)", y = "Value x_t")

上の2本は、同じ高さのまわりを同じ幅で動きます。ランダムウォークは、高さが定まりません。

3本の標本自己相関関数（autocorrelation function, ACF）を出します。
青い破線は、ホワイトノイズなら95%の確率でこの中に入る、という範囲（$\pm 1.96/\sqrt{n}$）です。

In [ ]:
sim |>
  ACF(x, lag_max = 30) |>
  autoplot() +
  labs(x = "Lag h (step)", y = "Sample ACF")

- ホワイトノイズ：ほぼすべてのラグで破線の中。
- AR(1)：ラグとともに速く小さくなる。
- ランダムウォーク：ほとんど小さくならない。

AR(1)の理論ACFは$\rho(h) = \phi^h$です。標本ACF（棒）に理論値（赤い点）を重ねます。

In [ ]:
ar1 <- sim |> filter(series == "2 AR(1)")
theory <- data.frame(lag = 1:20, acf = 0.7^(1:20))
ar1 |>
  ACF(x, lag_max = 20) |>
  autoplot() +
  geom_point(data = theory, aes(lag, acf), colour = "red", size = 2) +
  labs(x = "Lag h (step)", y = "ACF (bar: sample, red: theory)")

偏自己相関関数（partial autocorrelation function, PACF）は、あいだの時点の影響を取り除いた相関です。
AR(1)では、理論上ラグ1だけが$\phi$で、ラグ2以降は0です。

In [ ]:
ar1 |>
  PACF(x, lag_max = 20) |>
  autoplot() +
  labs(x = "Lag h (step)", y = "Sample PACF")

ラグ1だけが0.7の近くにあり、ラグ2以降はほぼ破線の中です。

### A-3. 標本平均のばらつき

AR(1)の標本平均$\bar{x}$の分散は、$n$が大きいとき、次の式で近似できます。

$$
\mathrm{Var}(\bar{x}) \approx \frac{\gamma(0)}{n} \cdot \frac{1+\phi}{1-\phi}
$$

$\gamma(0)$は系列の分散です。iidのときの式$\gamma(0)/n$の、$(1+\phi)/(1-\phi)$倍になります。

長さ200のAR(1)を2000本作り、標本平均の分散を実際に測って、式と比べます。

In [ ]:
set.seed(1)
n <- 200
sim_ar1 <- function(n, phi) {           # AR(1)を1本作る（最初の100個は捨てる）
  x <- stats::filter(rnorm(n + 100), phi, method = "recursive")
  x[-(1:100)]
}
res <- lapply(c(0, 0.5, 0.9), function(phi) {
  xbar <- replicate(2000, mean(sim_ar1(n, phi)))
  gamma0 <- 1 / (1 - phi^2)             # 系列の分散（ノイズの分散は1）
  data.frame(
    phi = phi,
    var_sim = var(xbar),                # シミュレーションで測った分散
    var_iid = gamma0 / n,               # iidだと思ったときの式
    var_theory = gamma0 / n * (1 + phi) / (1 - phi),   # 従属性を入れた式
    n_eff = n * (1 - phi) / (1 + phi)   # 実効サンプルサイズ
  )
})
round(do.call(rbind, res), 4)

- `var_sim`は`var_theory`に近く、`var_iid`とは合いません（$\phi=0$のときだけ3つが一致します）。
- $\phi = 0.9$では、200個のデータに、独立なデータ約10個分の情報しかありません。これを実効サンプルサイズ（effective sample size）と呼びます。

### A-4. 見せかけの回帰に戻る

ランダムウォークの差分（difference）$x_t - x_{t-1}$は、もとのホワイトノイズです。
差分をとってから回帰すると、割合がどうなるかを見ます。

In [ ]:
set.seed(1)
p_diff <- replicate(1000, {
  x <- cumsum(rnorm(200)); y <- cumsum(rnorm(200))
  summary(lm(diff(y) ~ diff(x)))$coefficients[2, 4]
})
mean(p_diff < 0.05)

ほぼ5%に戻ります。

## B. 実データを解析する

`tsibbledata`パッケージの`gafa_stock`を使います。米国の4社（Apple・Amazon・Facebook・Google）の株価です。

### B-0. 設定

このセルの値を変えると、B節の解析がすべて変わります。まずはこのまま実行してください。

In [ ]:
# ここを変えると、以下の解析がすべて変わる
symbol    <- "GOOG"   # 銘柄："AAPL" "AMZN" "FB" "GOOG" のどれか
year_from <- 2014     # 始まりの年：2014〜2018
year_to   <- 2018     # 終わりの年：year_from 以上、2018 以下
lag_max   <- 30       # ACFを見る最大のラグ：10〜50

### B-1. データを確かめる

In [ ]:
glimpse(gafa_stock)
gafa_stock |>
  as_tibble() |>
  group_by(Symbol) |>
  summarise(n = n(), from = min(Date), to = max(Date))

- 1行は、1社の1取引日です。`Close`が終値（米ドル）です。
- 期間は2014年1月2日から2018年12月31日、各社1258取引日です。

### B-2. 前処理

銘柄と期間を絞ります。

In [ ]:
stock_raw <- gafa_stock |>
  filter(Symbol == symbol,
         format(Date, "%Y") >= year_from, format(Date, "%Y") <= year_to)
interval(stock_raw)
nrow(stock_raw)

`interval()`の結果が`!`になっています。これは、間隔が一定でないという印です。
市場は土日と祝日に休むので、日付で並べると、ところどころに穴があきます。
このままではACFを計算できません。

そこで、取引日の通し番号を時間の軸にします。あわせて、対数収益率（log return）を作ります。

$$
r_t = 100 \times \left( \log P_t - \log P_{t-1} \right)
$$

$P_t$は終値で、$r_t$は前の取引日からの変化率（%）にほぼ等しい量です。

In [ ]:
stock <- stock_raw |>
  mutate(day = row_number()) |>
  update_tsibble(index = day, regular = TRUE) |>
  mutate(ret = 100 * difference(log(Close)),   # 対数収益率（%）
         abs_ret = abs(ret)) |>                # その絶対値（%）
  filter(!is.na(ret))                          # 1行目は前の日がないので外す
interval(stock)
stock |> select(Date, day, Close, ret, abs_ret) |> head(4)

間隔が`1`になりました。1行目は、前の日がなく収益率を計算できないので、外してあります。

### B-3. 終値

In [ ]:
ggplot(stock, aes(Date, Close)) +
  geom_line() +
  labs(x = "Date", y = "Closing price (USD)", title = symbol)

In [ ]:
stock |>
  ACF(Close, lag_max = lag_max) |>
  autoplot() +
  labs(x = "Lag h (trading day)", y = "Sample ACF of closing price")

ACFがほとんど小さくなりません。A-2のランダムウォークと同じ形です。

### B-4. 収益率

In [ ]:
ggplot(stock, aes(Date, ret)) +
  geom_line() +
  labs(x = "Date", y = "Log return (%)", title = symbol)

In [ ]:
stock |>
  ACF(ret, lag_max = lag_max) |>
  autoplot() +
  labs(x = "Lag h (trading day)", y = "Sample ACF of log return")

収益率のACFは、ほとんどのラグで破線の中にあります。A-2のホワイトノイズに近い形です。

### B-5. 収益率の絶対値

収益率の絶対値は、その日にどれだけ大きく動いたかを表します。

In [ ]:
stock |>
  ACF(abs_ret, lag_max = lag_max) |>
  autoplot() +
  labs(x = "Lag h (trading day)", y = "Sample ACF of |log return|")

In [ ]:
stock |>
  PACF(abs_ret, lag_max = lag_max) |>
  autoplot() +
  labs(x = "Lag h (trading day)", y = "Sample PACF of |log return|")

収益率そのものには自己相関がほとんどないのに、絶対値には正の自己相関が長く残ります。無相関に近くても、iidではありません。
大きく動いた日のあとは、向きは分からないが、また大きく動きやすい、ということです。

### B-6. 平均の標準誤差

A-3の式を一般の定常過程に広げると、標本平均の分散は次のようになります。

$$
\mathrm{Var}(\bar{x}) \approx \frac{\gamma(0)}{n} \left\{ 1 + 2 \sum_{h=1}^{\infty} \rho(h) \right\}
$$

波かっこの中を`factor`と呼びます。実データでは$\rho(h)$が分からないので、標本ACFで置き換え、ラグ$L$（`lag_max`）までで打ち切ります。
遠いラグの標本ACFは誤差が大きいので、重み$1 - h/(L+1)$をかけて、遠いラグほど小さく足します。

$$
\widehat{\mathrm{factor}} = 1 + 2 \sum_{h=1}^{L} \left(1 - \frac{h}{L+1}\right) \hat{\rho}(h)
$$

`n_eff = n / factor`が実効サンプルサイズ、`se_iid`がiidだと思ったときの標準誤差、`se_dep`が従属性を入れた標準誤差です。

In [ ]:
ess_row <- function(name) {
  x <- stock[[name]]
  n <- length(x)
  rho <- acf(x, lag.max = lag_max, plot = FALSE)$acf[-1]
  weight <- 1 - (1:lag_max) / (lag_max + 1)   # 遠いラグほど小さい重み
  factor <- 1 + 2 * sum(weight * rho)
  data.frame(series = name, n = n, mean = mean(x), rho1 = rho[1],
             factor = factor, n_eff = n / factor,
             se_iid = sd(x) / sqrt(n), se_dep = sd(x) / sqrt(n / factor))
}
ess <- do.call(rbind, lapply(c("Close", "ret", "abs_ret"), ess_row))
ess[, -1] <- round(ess[, -1], 3)
ess

読み方：

- `ret`（収益率）：`factor`は1の前後です。自己相関がほとんどないので、いつもの標準誤差とあまり変わりません。標本ACFには誤差があるので、`factor`が1より小さく出ることもあります。
- `abs_ret`（絶対値）：`factor`が1より大きく、`n_eff`は`n`よりずっと小さくなります。標準誤差は`se_iid`より広くとる必要があります。
- `Close`（終値）：`n_eff`は極端に小さくなります。ただし、終値は定常ではないので、この式の前提が成り立っていません。平均そのものに意味がない、と読みます。

## C. 設定を変えて回し直す

B-0の設定セルを書き替えて、B節を上からすべて実行し直してください
（Colabでは、設定セルを選んで、メニューの［ランタイム］→［以降のセルを実行］）。

変えられる値：

| 変数 | 値 |
| --- | --- |
| `symbol` | `"AAPL"` `"AMZN"` `"FB"` `"GOOG"` |
| `year_from` | 2014〜2018 |
| `year_to` | `year_from`以上、2018以下 |
| `lag_max` | 10〜50 |

読むときの観点：

1. 終値のACFは、銘柄や期間を変えても「ほとんど小さくならない」ままか。
2. 収益率のACFが破線の外に出るラグはあるか。それは銘柄や期間を変えても同じラグか。
3. 絶対値の`factor`と`n_eff`は、どの銘柄・どの期間で大きく変わるか。
4. 期間を1年に縮めると、破線の幅と`se_dep`はどう変わるか。

正解は1つではありません。自分の設定で出た結果を、最初の設定（GOOG、2014〜2018年）の結果と比べてください。